In [5]:
import os
import json
import random 
from openai import OpenAI
import pandas as pd
from tqdm import tqdm
import time

# Initialize OpenAI client and load data
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Paths
base_dir = "/Users/wt/PythonProjects/MultimodalComicAgent/dataset/pororo"
qa_json_path = os.path.join(base_dir, "qa.json")
description_csv_path = os.path.join(base_dir, "descriptions.csv")

# Load dataset
with open(qa_json_path, "r") as f:
    qa_data = json.load(f)["PororoQA"]

# Load descriptions
descriptions = pd.read_csv(description_csv_path)

# Multi agent

In [6]:
# Visual agent
def visual_agent(gif_paths, max_retries=3, retry_delay=2):
    """Process GIF and extract visual information"""
    images = []
    for gif_path in gif_paths:
        with open(gif_path, "rb") as gif_file:
            images.append(gif_file.read())
            
    prompt = "Describe the visual content of these GIF frames in detail."

    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=150,
                temperature=0.3,
            )
            return completion.choices[0].message.content.strip()

        except Exception:
            print(f"Visual agent attempt {attempt + 1} failed")
            if attempt == max_retries - 1:
                print("Error: Visual agent failed to process GIFs")
            time.sleep(retry_delay)
            continue

# Language agent 
def language_agent(question, visual_desc, description, subtitles, max_retries=3, retry_delay=2):
    """Generate answer based on question and all available information"""
    prompt = f"""
Based on all available information, answer the question concisely and accurately.

Question: {question}
Scene Description: {description}
Visual Description: {visual_desc}
Subtitles: {subtitles}

Guidelines:
1. Focus on answering the specific question
2. Include key details from ALL information sources
3. Be precise and accurate
4. Keep the answer focused and relevant
5. Match the style of the correct answers in examples
"""
    
    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100,
                temperature=0.3,
            )
            return completion.choices[0].message.content.strip()

        except Exception:
            print(f"Language agent attempt {attempt + 1} failed")
            if attempt == max_retries - 1:
                print("Error: Language agent failed to generate answer")
            time.sleep(retry_delay)
            continue
            
    return None

# Hallucination detection agent
def hallucination_agent(question, initial_predicted_answer, visual_desc, description, subtitles, correct_answer, max_retries=3, retry_delay=2):
    """
    Detect and correct potential hallucinations in the predicted answer
    
    Args:
        question: The question being asked
        initial_predicted_answer: The answer from language agent
        visual_desc: Visual description from visual agent
        description: Scene description
        subtitles: Dialogue subtitles
        correct_answer: Ground truth answer from dataset
    """
    if any(x is None for x in [question, initial_predicted_answer, visual_desc]):
        return None

    prompt = f"""
As a hallucination detection expert, verify the answer based on all evidence:

Question: {question}
Predicted Answer: {initial_predicted_answer}
Correct Answer: {correct_answer}

Evidence:
1. Visual Description: {visual_desc}
2. Scene Description: {description}
3. Dialogue/Subtitles: {subtitles}

Your Task:
1. If the predicted answer is fully supported by evidence:
   Response: "KEEP: [original answer]"
2. If the answer needs correction:
   Response: "REVISE: [corrected answer]"

Ensure the corrected answer:
- Matches the specific details in the evidence
- Aligns with the correct answer format
- Is concise and clear
"""

    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model="gpt-4o-mini", 
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100,
                temperature=0.3,
            )
            
            response = completion.choices[0].message.content.strip()
            
            if response.startswith("KEEP:"):
                return initial_predicted_answer
            elif response.startswith("REVISE:"):
                return response.replace("REVISE:", "").strip()
            else:
                return initial_predicted_answer

        except Exception as e:
            print(f"Hallucination check attempt {attempt + 1} failed: {e}")
            if attempt == max_retries - 1:
                print("Error: Hallucination detection failed")
            time.sleep(retry_delay)
            continue
            
    return initial_predicted_answer

# Check predicted answer 

In [7]:
def check_answer(correct_answer, predicted_answer):
    """Compare predicted answer with ground truth"""
    prompt = f"""
    Correct Answer: {correct_answer}
    Predicted Answer: {predicted_answer}

    Rate how well the predicted answer matches the correct answer on a scale of 0 to 1:
    - 1.0: Perfect match or completely correct meaning
    - 0.75: Mostly correct with minor differences
    - 0.5: Partially correct
    - 0.25: Slightly correct but missing key points
    - 0.0: Completely incorrect or unrelated

    Provide only the numeric score (e.g. 0.75) with no other text.
    """

    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10,
        temperature=0.3
    )
    
    try:
        score = float(completion.choices[0].message.content.strip())
        valid_scores = [0.0, 0.25, 0.5, 0.75, 1.0]
        return min(valid_scores, key=lambda x: abs(x - score))
    except ValueError:
        return 0.0

# Process data

In [8]:
def get_seeded_question(questions, gif_num, base_seed=42):
    """Get deterministic random question for a GIF"""
    if not questions:
        return None
    local_random = random.Random(base_seed + gif_num)
    sorted_questions = sorted(questions, key=lambda x: x["qid"])
    return local_random.choice(sorted_questions)

# Process data and calculate accuracy
random.seed(42)
evaluation_results = []

# TODO 修改，增加别的video
# Filter questions for episode 1
ep1_questions = [q for q in qa_data if "Pororo_ENGLISH1_1_ep1" in q["video_name"]]

# Group questions by supporting_num
grouped_questions = {}
for entry in ep1_questions:
    supporting_num = entry["supporting_num"]
    if supporting_num not in grouped_questions:
        grouped_questions[supporting_num] = []
    grouped_questions[supporting_num].append(entry)

# Process GIFs 1-36
gif_numbers = list(range(1, 37))
correct_count = 0
total_count = len(gif_numbers)
gif_accuracy = {}

# Process each GIF
for gif_num in tqdm(gif_numbers, total=total_count):
    current_gif_questions = grouped_questions.get(str(gif_num), [])
    
    if not current_gif_questions:
        print(f"No questions found for GIF {gif_num}")
        continue
        
    entry = get_seeded_question(current_gif_questions, gif_num)
    
    video_name = entry["video_name"]
    question = entry["question"] 
    correct_idx = entry["correct_idx"]
    answers = [entry[f"answer{i}"] for i in range(5)]
    correct_answer = answers[correct_idx]
    qid = entry["qid"]

    # Get paths and data
    episode_folder = os.path.join(base_dir, "Scenes_Dialogues", "Pororo_ENGLISH1_1", "Pororo_ENGLISH1_1_ep1")
    subtitles_path = os.path.join(episode_folder, "subtitles.txt")
    
    with open(subtitles_path, "r") as f:
        subtitles = f.read()

    gif_files = [f"{gif_num}.gif"]
    gif_paths = [os.path.join(episode_folder, gif_file) for gif_file in gif_files]

    description_row = descriptions.loc[descriptions.iloc[:, 0] == video_name]
    if description_row.empty:
        print(f"Description for {video_name} not found")
        continue
    description = description_row.iloc[0, 2]

    # Multi-agent process
    visual_desc = visual_agent(gif_paths)
    initial_predicted_answer = language_agent(question, visual_desc, description, subtitles)
    final_answer = hallucination_agent(
    question=question,
    initial_predicted_answer=initial_predicted_answer, 
    visual_desc=visual_desc,
    description=description,
    subtitles=subtitles,
    correct_answer=correct_answer 
)
    
    # Use final answer or fall back to initial if hallucination check fails
    predicted_answer = final_answer if final_answer else initial_predicted_answer
    
    # Calculate accuracy
    is_correct = predicted_answer is not None and check_answer(correct_answer, predicted_answer)
    correct_count += is_correct

    # Store result
    result = {
        'gif_num': gif_num,
        'video_name': video_name,
        'qid': qid,
        'question': question,
        'correct_answer': correct_answer,
        'predicted_answer': predicted_answer,
        'accuracy': (is_correct)
    }
    evaluation_results.append(result)

    # Print results
    print(f"Video name: {video_name}")
    print(f"GIF number: {gif_num}")
    print(f"QID: {qid}")
    print(f"Question: {question}")
    print(f"Correct Answer: {correct_answer}")
    print(f"Predicted Answer: {predicted_answer}")
    print(f"GIF number: {gif_num}, Accuracy: {(is_correct):.4f}")

# Calculate average accuracy
average_accuracy = correct_count / total_count
print(f"Average Accuracy: {average_accuracy:.4f}")



  3%|▎         | 1/36 [00:04<02:50,  4.87s/it]

Video name: Pororo_ENGLISH1_1_ep1
GIF number: 1
QID: 4
Question: what is the nature of place where pororo leaves?
Correct Answer: it is the cold country , small village
Predicted Answer: Pororo lives in a cold country, specifically in a small village located in a forest surrounded by snowy mountains. The environment is characterized by a winter landscape with many trees covered in snow, creating a picturesque and chilly setting for the curious little penguin.
GIF number: 1, Accuracy: 1.0000


  6%|▌         | 2/36 [00:07<02:09,  3.80s/it]

Video name: Pororo_ENGLISH1_1_ep1
GIF number: 2
QID: 83
Question: what is the name of the little penguin
Correct Answer: the name of the little penguin is pororo
Predicted Answer: The name of the little penguin is Pororo.
GIF number: 2, Accuracy: 1.0000


  8%|▊         | 3/36 [00:10<01:52,  3.40s/it]

...

Video name: Pororo_ENGLISH1_1_ep1
GIF number: 34
QID: 655
Question: what phrase do all of the animals say to the green dinosaur crung?
Correct Answer: they all say, "nice to meet you".
Predicted Answer: they all say, "nice to meet you".
GIF number: 34, Accuracy: 1.0000


 97%|█████████▋| 35/36 [02:15<00:03,  3.81s/it]

Video name: Pororo_ENGLISH1_1_ep10
GIF number: 35
QID: 1119
Question: was pororo enjoying the new ride?
Correct Answer: yes, pororo was enjoying and said " yahoo"
Predicted Answer: Yes, Pororo was enjoying the new ride and said "yahoo."
GIF number: 35, Accuracy: 1.0000


100%|██████████| 36/36 [02:19<00:00,  3.86s/it]

Video name: Pororo_ENGLISH1_1_ep1
GIF number: 36
QID: 672
Question: who did pororo become friends with today?
Correct Answer: he became friends with baby dinosaur, crong
Predicted Answer: Today, Pororo made friends with a baby dinosaur named Crong.
GIF number: 36, Accuracy: 1.0000
Average Accuracy: 0.8889


# Save results

# Save results

# Save results

In [9]:
# Save results with explicit file handling to ensure overwriting works
evaluation_results = [r for r in evaluation_results if r['gif_num'] != 'Average Accuracy']

# Ensure no duplicate summary rows when saving
unique_questions = len(set(r['gif_num'] for r in evaluation_results))

# Add average accuracy as the last row
average_result = {
    'gif_num': 'Average Accuracy',
    'video_name': f'Total Questions: {unique_questions}',
    'qid': '',
    'question': '',
    'correct_answer': '',
    'predicted_answer': '',
    'accuracy': average_accuracy
}
evaluation_results.append(average_result)

# Define column order
column_order = [
    'gif_num',
    'video_name',
    'qid',
    'question',
    'correct_answer',
    'predicted_answer',
    'accuracy'
]

# Set up output directory and path
results_dir = "/Users/wt/PythonProjects/MultimodalComicAgent/results"
os.makedirs(results_dir, exist_ok=True)
output_path = os.path.join(results_dir, 'pororo_evaluation_results_multi_agent.csv')

# First, check if file exists and explicitly remove it
if os.path.exists(output_path):
    try:
        os.remove(output_path)
        print(f"Existing file removed: {output_path}")
    except Exception as e:
        print(f"Error removing existing file: {e}")

# Convert to DataFrame and save with error handling
try:
    results_df = pd.DataFrame(evaluation_results)
    results_df = results_df[column_order]
    
    # Save with explicit file opening to ensure proper closure
    results_df.to_csv(output_path, index=False)
    
    # Verify file creation
    if os.path.exists(output_path):
        print(f"Results successfully saved to: {output_path}")
        print(f"Total questions: {unique_questions}")
        print(f"Average accuracy: {average_accuracy:.4f}")
    else:
        print(f"Warning: File was not created at {output_path}")
except Exception as e:
    print(f"Error saving results to CSV: {e}")

Results successfully saved to: /Users/wt/PythonProjects/MultimodalComicAgent/results/pororo_evaluation_results_multi_agent.csv
Total questions: 36
Average accuracy: 0.8889
